# Negotiation · 모델 사용 설정 튜닝 (Colab T4)

**GPU 미실행 (2026-11-18 기준, CPU 검증만).** 이 노트는 아직 Colab이나 다른 GPU에서 한 번도 실행하지 않았습니다. CPU에서 키워드 기준 backend와 테스트 대역으로 셀의 흐름(계획 → 짧은 확인 → 전체 실행 → 리더보드 → 확인 구간 → 상태 ZIP)만 확인했습니다. 그래서 저장된 출력이 없습니다. 실제 GPU에서 끝까지 실행해 확인한 뒤에 이 문구를 고칩니다.

여기서 '튜닝'은 모델을 학습시키는 일이 아닙니다. 고정된 Zoro-1 판단 모델이 내는 두 추정값(후보 수락 가능성, 지금 조건이 최종안인지)을 협상 정책이 **어떻게 쓸지** 미리 정한 8개 구성 중에서 고릅니다. 모델, 질문 문구, 정책 상수는 바꾸지 않습니다.

- **사용하는 사례:** 합성 개발 분할과 보정 분할뿐입니다. 평가 분할은 이 노트에서 사용하지 않으며, 실행기에도 그 분할을 지정하는 방법이 없습니다.
- **시간과 요청 수:** 이 노트와 실행기가 보여 주는 '분'과 '요청 수'는 모두 2026-11-16 T4 실행(요청당 약 1.75초)에서 가져온 **추정치**이며 측정값이 아닙니다.
- **결과의 범위:** 규칙으로 움직이는 합성 공급사를 상대로 한 개발 사례의 값입니다. 실제 협상 성과가 아니고 보류 평가 결과도 아닙니다.
- **실패한 추정:** 모델 호출이 실패하거나 중단되면 그 값은 저장하지 않고, 채워 넣지 않고, 자동으로 다시 시도하지 않습니다.


## 실행 순서

먼저 **파일 → Drive에 사본 저장**, **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요. 소스 ZIP, 그 ZIP의 내용 해시(64자리), 본인 계정의 Zoro-1 모델 접근 권한이 필요합니다. 왼쪽 ▶를 위에서 아래로 누릅니다.

| 셀 | 하는 일 | 모델 호출 |
|---|---|---|
| 1 | 소스 ZIP 확인, 상태 폴더 준비(이전 상태 ZIP이 있으면 복원) | 없음 |
| 2~4 | 도구 설치, Hugging Face 로그인, 모델 다운로드·변환 | 없음 |
| 5 | 계획 기록: 프로토콜을 파일로 고정하고 추정 요청 수를 출력 | 없음 |
| 6 | 짧은 확인: 개발 사례 2건을 두 번 실행 | 있음 (적음) |
| 7 | 전체 실행: 시간 예산 안에서 진행하고 상태 ZIP을 만듦. 다시 누르면 이어서 진행 | 있음 |
| 8 | (선택) 공급사 발언 진단 | 있음 |
| 9 | 리더보드와 선택 결과 보기 | 없음 |
| 10 | 확인 구간: 체크해야 실행되며 한 번만 쓸 수 있음 | 있음 |
| 11 | 상태 ZIP 내려받기 | 없음 |

**런타임이 끊겨도 처음부터 다시 하지 않습니다.** 모델이 답한 요청은 상태 폴더에 바로 저장되고, 7번과 10번은 끝날 때마다 상태 ZIP을 만듭니다. 11번에서 ZIP을 내려받아 두었다가, 새 런타임에서 그 ZIP을 왼쪽 📁 파일 패널에 올리고 1번부터 다시 실행하면 저장된 답을 재사용해 이어서 진행합니다.

'모두 실행'을 눌러도 8번과 10번은 체크하지 않으면 실행되지 않습니다.


### 1. 소스 ZIP과 상태 폴더

소스 ZIP은 저장소 담당자가 개발 PC에서 `python scripts/make_source_bundle.py`로 만듭니다. 그 명령이 **마지막 줄에 출력하는 64자리 값**을 아래 셀 오른쪽의 `EXPECTED_CONTENT_SHA256`에 붙여 넣으세요. 값이 비어 있거나 ZIP 내용과 다르면 셀은 아무것도 풀지 않고 멈춥니다. 이 확인으로 어떤 소스로 실행했는지가 결과에 남습니다.

왼쪽 **📁 파일 → 세션 저장소에 업로드**로 `negotiation-source-<커밋 앞 12자리>.zip`을 올린 뒤 ▶를 누르세요. 파일이 없으면 셀 안에 파일 선택 버튼이 나타납니다. 소스 ZIP에는 가중치, 토큰, 캐시가 들어 있지 않습니다.

상태 폴더는 `/content/negotiation-tuning-state`이며 소스 폴더 밖에 있습니다. **이전에 내려받은 상태 ZIP**(`negotiation-tuning-state-….zip`)으로 이어서 하려면 이 셀을 실행하기 전에 그 ZIP **한 개**를 파일 패널에 올려 두세요. 모든 파일의 해시를 확인한 뒤 복원합니다. 상태 폴더가 이미 있으면 그대로 쓰고 ZIP은 복원하지 않습니다.


In [ ]:
#@title 1. 소스 ZIP 확인 · 상태 폴더 준비
EXPECTED_CONTENT_SHA256 = ''  #@param {type:"string"}
# NOT executed on a GPU as of 2026-11-18. This notebook was checked on CPU only, with the keyword
# reference backend and test doubles. Minutes and request counts it shows are estimates.
import hashlib, io, json, os, re, signal, stat, subprocess, sys, uuid, zipfile
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

NEG_READY = False          # the model is prepared (set by cell 4)
NEG_STATE_READY = False    # the source is verified and the state folder is prepared (set at the end of this cell)
NEG_CONTENT = Path('/content')
if not NEG_CONTENT.is_dir():
    raise RuntimeError('이 셀은 Google Colab에서 실행하세요.')
neg_expected = EXPECTED_CONTENT_SHA256.strip().lower()
if not neg_expected:
    raise ValueError('EXPECTED_CONTENT_SHA256이 비어 있습니다. 소스 ZIP을 만든 사람이 '
                     '`python scripts/make_source_bundle.py`의 마지막 줄에서 받은 64자리 값을 '
                     '오른쪽 입력란에 붙여 넣고 다시 실행하세요. 값을 확인하지 않은 ZIP은 풀지 않습니다.')
if not re.fullmatch(r'[0-9a-f]{64}', neg_expected):
    raise ValueError('EXPECTED_CONTENT_SHA256은 64자리 16진수여야 합니다. 앞뒤가 잘리지 않았는지 확인하세요.')
NEG_STATE = NEG_CONTENT / 'negotiation-tuning-state'    # outside the source folder
NEG_PACKS = NEG_CONTENT / 'negotiation-tuning-packs'    # state ZIPs made here; never overwritten
neg_state_zips = sorted(p for p in NEG_CONTENT.glob('negotiation-tuning-state*.zip') if p.is_file())
if not (NEG_STATE / 'protocol.json').is_file() and len(neg_state_zips) > 1:
    raise ValueError('파일 패널에 상태 ZIP이 여러 개 있습니다: ' + ', '.join(p.name for p in neg_state_zips)
                     + '. 이어서 할 ZIP 한 개만 남기고 다시 실행하세요.')


def neg_source_contents(data):
    # The same checks and the same content-hash formula as scripts/make_source_bundle.py.
    with zipfile.ZipFile(io.BytesIO(data)) as bundle:
        members = [m for m in bundle.infolist() if not m.is_dir()]
        roots = [m.filename[:-len('pyproject.toml')] for m in members
                 if m.filename == 'pyproject.toml' or m.filename.endswith('/pyproject.toml')]
        if len(roots) != 1 or sum(m.file_size for m in members) > 32 * 1024**2:
            raise ValueError('예상한 프로젝트 ZIP이 아닙니다.')
        prefix = roots[0]
        contents = {}
        for member in members:
            if not member.filename.startswith(prefix):
                raise ValueError('ZIP의 프로젝트 경로를 확인할 수 없습니다.')
            name = member.filename[len(prefix):]
            path = PurePosixPath(name)
            if (not name or '\\' in name or ':' in name or path.is_absolute()
                    or '..' in path.parts or name in contents
                    or stat.S_ISLNK(member.external_attr >> 16)):
                raise ValueError('ZIP에 허용하지 않는 경로가 있습니다.')
            contents[name] = bundle.read(member)
        comment = bundle.comment.decode('ascii', 'replace').strip()
    content_hash = hashlib.sha256(''.join(
        f'{n}\0{hashlib.sha256(contents[n]).hexdigest()}\n' for n in sorted(contents)
    ).encode()).hexdigest()
    return content_hash, contents, comment


def neg_pick_source(archives, expected):
    # Only an archive whose content hash equals the pasted value is accepted.
    seen = []
    for name, data in archives:
        try:
            content_hash, contents, comment = neg_source_contents(data)
        except Exception as error:
            seen.append(f'  {name}: 읽을 수 없음 ({type(error).__name__}: {error})')
            continue
        if content_hash == expected:
            return {'archive': name, 'content_sha256': content_hash, 'contents': contents,
                    'zip_sha256': hashlib.sha256(data).hexdigest(), 'zip_comment': comment}
        seen.append(f'  {name}: 내용 해시 앞 12자리 {content_hash[:12]}')
    raise ValueError('입력한 EXPECTED_CONTENT_SHA256(앞 12자리 ' + expected[:12] + ')과 내용이 같은 소스 ZIP이 '
                     '없습니다. ZIP과 해시를 같은 `make_source_bundle.py` 실행에서 받았는지 확인하세요.\n'
                     + '\n'.join(seen))


neg_found = sorted(p for p in NEG_CONTENT.glob('negotiation-source-*.zip')
                   if p.is_file() and p.stat().st_size <= 64 * 1024**2)
if neg_found:
    print('파일 패널의 소스 ZIP을 확인합니다:', ', '.join(p.name for p in neg_found))
    neg_archives = [(p.name, p.read_bytes()) for p in neg_found]
else:
    from google.colab import files
    neg_uploaded = files.upload()
    if len(neg_uploaded) != 1:
        raise ValueError('소스 ZIP 한 개를 선택하세요. 파일 선택창이 열리지 않으면 왼쪽 파일 패널로 '
                         '업로드한 뒤 다시 실행하세요.')
    neg_archives = list(neg_uploaded.items())
    del neg_uploaded
neg_source = neg_pick_source(neg_archives, neg_expected)
del neg_archives
NEG_REPO = NEG_CONTENT / ('negotiation-tuning-src-' + uuid.uuid4().hex[:12])
NEG_REPO.mkdir()
for neg_name, neg_data in neg_source.pop('contents').items():
    neg_target = NEG_REPO / neg_name
    neg_target.parent.mkdir(parents=True, exist_ok=True)
    neg_target.write_bytes(neg_data)
del neg_name, neg_data, neg_target


def neg_stamp():
    return datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')


NEG_LOGS = NEG_CONTENT / 'negotiation-tuning-logs' / ('session-' + neg_stamp() + '-' + uuid.uuid4().hex[:6])
NEG_LOGS.mkdir(parents=True)
NEG_ENV = {'PYTHONPATH': os.pathsep.join([str(NEG_REPO / 'src'), str(NEG_REPO / 'negotiation/src')]),
           'PYTHONUNBUFFERED': '1', 'PYTHONIOENCODING': 'utf-8'}
(NEG_LOGS / 'source.json').write_text(json.dumps({
    **neg_source, 'expected_content_sha256': neg_expected, 'created_at': datetime.now(timezone.utc).isoformat(),
    'notice': 'Tuning notebook session. Not executed on a GPU as of 2026-11-18 unless a later result says so. '
              'zip_comment is whatever the archive carries (git archive writes the commit id); it is not verified.'
}, indent=2), encoding='utf-8')


def neg_stop(process):
    # After an interrupt: let the command finish its current request, then ask it, then force it.
    print('중단 요청을 받았습니다. 실행 중인 명령이 정리하고 끝나기를 기다립니다.', flush=True)
    for action, seconds in ((None, 10), ('interrupt', 30), ('terminate', 15)):
        try:
            if action == 'interrupt':
                process.send_signal(signal.SIGINT)
            elif action == 'terminate':
                process.terminate()
        except (OSError, ValueError):
            pass
        try:
            rest, _ = process.communicate(timeout=seconds)
            return process.returncode, rest or ''
        except (subprocess.TimeoutExpired, KeyboardInterrupt):
            continue
    process.kill()
    try:
        rest, _ = process.communicate(timeout=15)
    except (subprocess.TimeoutExpired, KeyboardInterrupt):
        rest = ''
    return process.poll(), rest or ''


def neg_run(*args, check=True):
    # One repository command with a log file. Returns {'exit_code', 'lines', 'log'}.
    command = [sys.executable, '-u', *map(str, args)]
    log = NEG_LOGS / ('command-' + uuid.uuid4().hex[:8] + '.log')
    print('실행:', ' '.join(command), flush=True)
    # The environment may contain HF_TOKEN; never serialize or print it.
    env = {**os.environ, **NEG_ENV}
    lines, interrupted = [], False
    with log.open('w', encoding='utf-8') as output:
        process = subprocess.Popen(command, cwd=NEG_REPO, env=env,
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding='utf-8', errors='replace')
        try:
            for line in process.stdout:
                output.write(line); output.flush()
                print(line, end='', flush=True)
                lines.append(line.rstrip('\n'))
            exit_code = process.wait()
        except KeyboardInterrupt:
            interrupted = True
            exit_code, rest = neg_stop(process)
            output.write(rest)
            print(rest, end='', flush=True)
            lines.extend(rest.splitlines())
    with (NEG_LOGS / 'commands.jsonl').open('a', encoding='utf-8') as history:
        history.write(json.dumps({'argv': command, 'exit_code': exit_code, 'log': log.name,
                                  'interrupted': interrupted}) + '\n')
    if interrupted:
        raise KeyboardInterrupt
    if exit_code and check:
        raise RuntimeError(f'실행 실패(exit={exit_code}). 위 오류와 {log.name}을 확인하세요.')
    return {'exit_code': exit_code, 'lines': lines, 'log': log.name}


NEG_STATUS_TEXT = {
    'complete': '완료했습니다.',
    'skipped_complete': '이미 완료된 단계라 다시 실행하지 않았습니다.',
    'partial_budget': '시간 예산에 도달해 정리하고 멈췄습니다. 같은 셀을 다시 실행하면 저장된 답을 재사용해 '
                      '이어서 진행합니다.',
    'pending_maps': '보정 맵이 아직 없어 일부 구성을 실행하지 못했습니다. 7번(전체 실행)을 실행하세요.',
    'failed': '추정에 실패한 구성이 있습니다. 실패한 값은 저장하지도 채우지도 않았고, 자동으로 다시 시도하지 '
              '않습니다. 실행은 그 자리에서 멈췄고 오류 원문은 위 로그에 있습니다. 같은 셀을 다시 실행하면 그 '
              '구성은 실패로 둔 채 나머지를 이어 갑니다. 다시 물으려면 맨 아래 "막혔을 때"의 재시도 항목을 따르세요.',
    'smoke_failed': '짧은 확인이 실패했습니다. 다음 단계로 진행하지 말고 로그와 상태 ZIP을 담당자에게 전달하세요.',
    'interrupted': '중단했습니다. 중단된 요청의 값은 저장하지 않았습니다. 다시 실행하면 이어서 진행합니다.',
    'initialization_failed': '모델 backend를 만들지 못했습니다. 2~4번을 다시 확인한 뒤 같은 셀을 다시 실행하세요.',
    'store_identity_mismatch': '지금 런타임이 저장된 답을 만든 런타임과 다릅니다(GPU·dtype·라이브러리). 서로 다른 '
                               '런타임의 답은 섞지 않습니다. 새 상태 폴더로 시작하세요("막혔을 때" 참고).',
    'runner_error': '실행기 자체가 실패했습니다. 로그와 상태 ZIP을 담당자에게 전달하세요.',
    'store_write_failed': '모델 답을 저장하지 못했습니다(디스크 공간 부족 등). 그 뒤로는 모델을 호출하지 않았습니다. '
                          '공간을 확보한 뒤 같은 셀을 다시 실행하세요.',
    'refused': '실행기가 이 명령을 거절했습니다. 거절이 보호하는 파일은 바뀌지 않았습니다.',
}


def neg_tune_summary(result):
    # Read the runner's own status lines; nothing is inferred from the exit code alone.
    summary = {'exit_code': result['exit_code'], 'log': result['log'], 'stage': None, 'status': None,
               'live_requests': None, 'selection': None, 'selected': None, 'confirm_verdict': None,
               'counts_as_model_evidence': None, 'refused': None}
    for line in result['lines']:
        found = re.match(r'tune run: stage=(\S+) status=(\S+) live_requests_this_process=(\d+)', line)
        if found:
            summary.update(stage=found.group(1), status=found.group(2), live_requests=int(found.group(3)))
        found = re.match(r'leaderboard: selection=(\S+)', line)
        if found:
            summary['selection'] = found.group(1)
            for key in ('selected', 'confirm_verdict', 'counts_as_model_evidence'):
                value = re.search(' ' + key + r'=(\S+)', line)
                summary[key] = value.group(1) if value else None
        if line.startswith('Refused: '):
            summary['refused'] = line[len('Refused: '):]
    if summary['status'] is None and summary['refused'] is not None:
        summary['status'] = 'refused'
    return summary


def neg_tune(command, *options):
    # One run_tune.py command on the state folder. Only `run` calls the model.
    if not NEG_STATE_READY:
        raise RuntimeError('1번 셀(소스 ZIP 확인 · 상태 폴더 준비)을 오류 없이 끝낸 뒤 실행하세요.')
    if command == 'run' and not NEG_READY:
        raise RuntimeError('2~4번의 모델 준비를 먼저 완료하세요.')
    backend = list(NEG_ARGS) if NEG_READY else []
    return neg_tune_summary(neg_run('negotiation/run_tune.py', command, '--state-dir', NEG_STATE,
                                    *options, *backend, check=False))


def neg_checked(summary, what):
    # Explain the runner's status in Korean, then stop the cell when the command did not succeed.
    if summary['status'] is not None:
        print(f"[{what}] status={summary['status']}:",
              NEG_STATUS_TEXT.get(summary['status'], '실행기 출력을 확인하세요.'))
    if summary['refused'] is not None:
        print('거절 사유:', summary['refused'])
    if summary['live_requests'] is not None:
        print('이번 실행에서 모델에 새로 보낸 요청 수:', summary['live_requests'], '(저장된 답 재사용은 제외)')
    if summary['selection'] == 'selected':
        print('선택된 구성:', summary['selected'], '(9번 셀에서 표로 확인하세요)')
    elif summary['selection'] is not None:
        print('선택이 아직 확정되지 않았습니다. 9번 셀에서 사유를 확인하세요.')
    if summary['confirm_verdict'] is not None:
        print('확인 구간 판정:', summary['confirm_verdict'])
    if summary['counts_as_model_evidence'] == 'False':
        print('주의: 이 상태의 결과는 모델 근거로 세지 않습니다(CPU 대역 또는 축소 블록).')
    if summary['exit_code']:
        raise RuntimeError(f"{what}: 끝나지 않았습니다(status={summary['status']}, exit={summary['exit_code']}). "
                           f"위 설명과 로그 {summary['log']}를 확인하세요.")
    return summary


def neg_pack():
    # ZIP of the state folder with a manifest (CPU only). Every pack gets a new name.
    if not (NEG_STATE / 'protocol.json').is_file():
        print('상태 폴더에 아직 계획이 없어 상태 ZIP을 만들지 않았습니다.')
        return None
    NEG_PACKS.mkdir(exist_ok=True)
    target = NEG_PACKS / ('negotiation-tuning-state-' + neg_stamp() + '-' + uuid.uuid4().hex[:4] + '.zip')
    neg_run('negotiation/run_tune.py', 'pack', '--state-dir', NEG_STATE, '--output', target)
    print('상태 ZIP:', target, '|', round(target.stat().st_size / 1024), 'KB')
    return target


def neg_stage(stage, *options, pack=True):
    # One `run --stage` call. The state is packed afterwards, also after a stop, an error or an interrupt.
    if not NEG_STATE_READY or not NEG_READY:
        return neg_tune('run')            # raises the matching explanation; nothing is run
    try:
        summary = neg_tune('run', '--stage', stage, *options)
    finally:
        if pack:
            try:
                neg_pack()
            except Exception as error:
                print('상태 ZIP을 만들지 못했습니다:', type(error).__name__, '- 11번 셀에서 다시 시도하세요.')
    return neg_checked(summary, stage)


def neg_failed_waiting():
    # Candidates on SELECT whose last attempt failed an estimate and that have no complete attempt.
    # Read from the attempt records themselves: an explicit `board` would accept such a failure and select.
    if (NEG_STATE / 'selection.json').is_file():
        return []
    waiting = []
    for folder in sorted((NEG_STATE / 'runs' / 'select').glob('*')):
        results = []
        for file in sorted(folder.glob('attempt-*/result.json')):
            try:
                results.append(json.loads(file.read_text(encoding='utf-8')))
            except ValueError:
                results.append({'status': 'unreadable'})
        if not any((r.get('config') or {}).get('role') == 'candidate' for r in results):
            continue
        if any(r.get('complete') is True and r.get('status') == 'complete' for r in results):
            continue
        if results[-1].get('status') == 'failed' and results[-1].get('reason') != 'map_unavailable':
            waiting.append(f"{folder.name} ({results[-1].get('reason')})")
    return waiting


def neg_show_board():
    # Show the files the runner wrote. No model call and no new number is computed here.
    from IPython.display import Markdown, display
    board_file = NEG_STATE / 'leaderboard.md'
    if not board_file.is_file():
        print('아직 리더보드가 없습니다. 7번(전체 실행)을 먼저 실행하세요.')
        return None
    display(Markdown(board_file.read_text(encoding='utf-8')))
    selection_file = NEG_STATE / 'selection.json'
    if not selection_file.is_file():
        print('선택 결과가 아직 없습니다(selection.json 없음). 위 리더보드의 "선택 보류" 사유를 확인하세요.')
        return None
    selection = json.loads(selection_file.read_text(encoding='utf-8'))
    chosen = selection['selected']
    print('선택 결과 (selection.json):')
    print(json.dumps({
        'id': chosen['id'], 'is_default': chosen['is_default'], 'estimates': chosen['estimates'],
        'mean_realized_saving_on_select': chosen['mean_realized_saving'],
        'evidence_class': chosen['evidence_class'],
        'selected_on': selection['block']['first_case_id'] + '..' + selection['block']['last_case_id'],
        'rehearsal_blocks': selection['rehearsal_blocks'], 'created_at_utc': selection['created_at_utc'],
    }, ensure_ascii=False, indent=2))
    print('이 선택은 선택 구간 표본 안에서 고른 것입니다. 기본 설정보다 낫다고 말하려면 10번의 확인 구간이 '
          '필요합니다. 선택된 설정 파일:', NEG_STATE / 'selected_config.json')
    return selection


if (NEG_STATE / 'protocol.json').is_file():
    neg_state_note = '기존 상태 폴더를 그대로 사용합니다'
    if neg_state_zips:
        print('상태 폴더가 이미 있어, 올려 둔 상태 ZIP은 복원하지 않았습니다.')
elif neg_state_zips:
    neg_run('negotiation/run_tune.py', 'restore', '--state-dir', NEG_STATE, '--input', neg_state_zips[0])
    neg_state_note = '이전 상태를 복원했습니다 (' + neg_state_zips[0].name + ')'
else:
    neg_state_note = '새 상태로 시작합니다'
NEG_STATE_READY = True
print('소스 확인 완료:', neg_source['archive'], '| 내용 해시', neg_source['content_sha256'][:12] + '…')
print('소스 폴더:', NEG_REPO)
print('상태 폴더:', NEG_STATE, '|', neg_state_note)
print('로그 폴더:', NEG_LOGS)


### 2. T4와 실행 도구 준비

이미 설치된 CUDA PyTorch는 유지하고 저장소가 정한 패키지를 설치합니다. 웹 로그인은 2026-11-17에 T4에서 확인한 실행 노트와 같은 Hugging Face Hub 1.33.0으로 맞춥니다. GPU가 없으면 이 셀에서 멈춥니다. Colab이 재시작을 요구하면 재시작한 뒤 1번부터 다시 진행하세요(상태 폴더는 남아 있습니다).


In [ ]:
#@title 2. T4 확인 · 필요한 도구 설치
NEG_READY = False
neg_run('scripts/gpu_setup.py', '--steps', 'env,install', '--run-dir', NEG_LOGS / 'setup-install')
neg_run('-m', 'pip', 'install', '--quiet', 'huggingface_hub==1.33.0')
print('설치 완료. 다음 셀에서 본인 계정으로 모델 접근을 확인합니다.')


### 3. 본인 Hugging Face 계정으로 로그인

[Zoro-1 모델 페이지](https://huggingface.co/trillionlabs/Zoro-1)에서 모델 접근을 승인받은 계정이 필요합니다. 이미 로그인했거나 Colab의 🔑 보안 비밀에 `HF_TOKEN`을 넣어 두었다면 재사용합니다. 그렇지 않으면 표시되는 웹 링크에서 로그인·승인을 완료하세요. 토큰을 코드나 채팅에 적지 마세요.

이 셀은 정해진 한 문장만 출력하고 토큰이나 환경 변수는 출력하지 않습니다. 웹 로그인을 쓴 경우 로그인 화면이 셀 출력에 남으므로, 개인 사본을 공유하기 전에 이 셀의 출력을 지우세요.


In [ ]:
#@title 3. 웹 로그인 또는 저장된 인증 사용
NEG_READY = False
neg_token = os.environ.get('HF_TOKEN')
if not neg_token:
    try:
        from google.colab import userdata
        neg_token = userdata.get('HF_TOKEN')
    except Exception:
        pass
if neg_token:
    os.environ['HF_TOKEN'] = neg_token
else:
    from huggingface_hub import get_token, login
    if not get_token():
        login(add_to_git_credential=False)
    if not get_token():
        raise RuntimeError('웹 로그인을 완료한 뒤 이 셀을 다시 실행하세요.')
neg_token = None
print('인증 준비 완료. 모델 접근 권한은 다음 다운로드에서 확인됩니다.')


### 4. 고정된 Zoro-1 모델 준비

고정된 revision의 모델을 내려받고 공식 내보내기 도구로 변환합니다. 최초 실행은 시간이 걸리고, 이 런타임에 모델 캐시가 남아 있으면 재사용합니다. 실행 옵션은 **자체 Transformers 어댑터, FP16, 입력 2048토큰, 판단 온도 1**로 고정합니다. 저장된 답을 서로 비교할 수 있도록 실행기는 온도 1이 아니면 거절합니다.


In [ ]:
#@title 4. 모델 다운로드 · 변환
NEG_READY = False
NEG_CACHE = Path('/content/sessionguard/.cache')
if not NEG_CACHE.is_dir():
    NEG_CACHE = Path('/content/negotiation-model-cache')
neg_run('scripts/gpu_setup.py', '--steps', 'sources,model,export', '--dtype', 'fp16',
        '--work-dir', NEG_CACHE, '--run-dir', NEG_LOGS / 'setup-model')
NEG_PATHS = json.loads((NEG_REPO / '.cache/gpu_paths.json').read_text(encoding='utf-8'))
for key in ('original_checkpoint', 'exported_checkpoint', 'official_source_dir'):
    if not Path(NEG_PATHS[key]).is_dir():
        raise RuntimeError(f'모델 준비 경로가 없습니다: {key}')
NEG_ARGS = ['--backend', 'zoro', '--original-checkpoint', NEG_PATHS['original_checkpoint'],
            '--exported-checkpoint', NEG_PATHS['exported_checkpoint'],
            '--official-source-dir', NEG_PATHS['official_source_dir'],
            '--device', 'cuda:0', '--dtype', 'fp16', '--max-input-tokens', '2048',
            '--decision-temperature', '1']
NEG_READY = True
print('모델 파일 준비 완료. 다음 셀에서 계획을 기록합니다(모델 호출 없음).')


### 5. 계획 기록 (모델 호출 없음)

후보 구성, 사례 구간, 선택 규칙, 확인 규칙, 허용된 결과 문장을 `protocol.json`으로 상태 폴더에 먼저 고정합니다. 이미 있으면 내용이 지금 소스와 같은지 확인만 하고 바꾸지 않습니다. 모델이 답을 하나라도 내기 전에 규칙이 정해져 있어야 결과를 보고 규칙을 고치는 일을 막을 수 있습니다.

| 구간 | 사례 | 용도 |
|---|---|---|
| D0 | dev-1100..1129 (30건) | 짧은 확인과 회귀 확인만. 실제 모델이 이미 본 사례라 선택에 쓰지 않음 |
| 보정 | calibration-2000..2059 (60건) | 확률 맵 적합 전용. 절감액을 표시하거나 비교하지 않음 |
| 선택 | dev-1130..1249 (120건) | 8개 후보 중 고르기 |
| 확인 | dev-1250..1549 (300건) | 선택한 구성을 한 번만 확인 |

셀이 출력하는 단계별 요청 수와 분은 **추정치(상한)** 입니다. 등록된 구간 기준으로 전체 실행(7번)은 상한 약 79분, 확인 구간(10번)은 상한 약 45분으로 추정합니다. 같은 요청의 답은 재사용하므로 실제로는 더 적을 수 있지만, 아직 GPU에서 측정한 값은 없습니다.


In [ ]:
#@title 5. 계획 기록 · 추정 요청 수 보기 (모델 호출 없음)
neg_checked(neg_tune('plan'), '계획')
neg_protocol = json.loads((NEG_STATE / 'protocol.json').read_text(encoding='utf-8'))
print('프로토콜:', neg_protocol['protocol_version'], '| sha256', neg_protocol['protocol_sha256'][:12] + '…')
if neg_protocol.get('rehearsal_blocks'):
    print('주의: 이 상태 폴더는 축소 블록(리허설)으로 계획되었습니다. 등록된 프로토콜의 결과가 아니며 '
          '실제 모델 backend로는 실행할 수 없습니다.')
print('위의 요청 수와 분은 2026-11-16 T4 실행에서 가져온 추정치이며 측정값이 아닙니다.')


### 6. 짧은 확인

기본 구성(M0)으로 개발 사례 2건을 **두 번** 실행합니다. 첫 번째는 모델을 호출해 답을 저장하고, 두 번째는 저장된 답만으로 같은 결과가 나와야 합니다. 두 번째에 모델 호출이 한 번이라도 생기면 실패로 처리하고 멈춥니다. 이 확인의 절감액은 성능 결론으로 쓰지 않습니다. 모델을 처음 올리는 시간이 여기에 포함됩니다.


In [ ]:
#@title 6. 짧은 확인 (개발 사례 2건 × 2회)
neg_smoke = neg_stage('smoke', pack=False)
print('짧은 확인 통과. 7번에서 전체 실행을 시작할 수 있습니다.')


### 7. 전체 실행

차례대로 짧은 확인 → 회귀 확인(D0) → 보정 구간 실행 → 맵 적합 → 선택 구간(후보 8개와 CPU 기준) → 리더보드를 진행합니다. `BUDGET_MINUTES`(기본 45분)에 도달하면 사례 사이에서 정리하고 멈춥니다. **이 셀은 다시 실행해도 안전합니다.** 끝난 구성은 건너뛰고, 이미 답을 받은 요청은 모델을 다시 부르지 않습니다. `status=partial_budget`이 나오면 같은 셀을 다시 누르세요.

멈추든 끝나든 오류가 나든 마지막에 상태 ZIP을 새 이름으로 만듭니다(`/content/negotiation-tuning-packs`). Colab 사용 시간이 얼마 남지 않았다면 바로 11번에서 내려받으세요.

확인 구간은 이 셀에 포함되지 않습니다. 10번에서 따로, 한 번만 실행합니다.


In [ ]:
#@title 7. 전체 실행 (시간 예산 안에서 · 다시 실행하면 이어서 진행)
BUDGET_MINUTES = 45  #@param {type:"number"}
neg_budget = float(BUDGET_MINUTES)
if not 0 < neg_budget <= 720:
    raise ValueError('BUDGET_MINUTES는 0보다 크고 720 이하인 분 단위 숫자여야 합니다.')
neg_all = neg_stage('all', '--budget-minutes', neg_budget)
if neg_all['status'] == 'partial_budget':
    print('아직 끝나지 않았습니다. 이 셀을 다시 실행하면 이어서 진행합니다.')
else:
    print('전체 실행이 끝났습니다. 9번에서 리더보드와 선택 결과를 확인하세요.')


### 8. (선택) 공급사 발언 진단

같은 협상 상태에서 공급사의 마지막 문장만 바꿔 가며(최종안이라고 말함, 같은 뜻의 다른 표현, 부정문, 지시문 삽입 등) 최종안 여부 추정값이 문장에 반응하는지 봅니다. 전체는 개발 상태 16개 × 문장 9개 = 144요청으로, 추정 약 4분입니다(측정값 아님).

이 진단을 실제 모델로 실행하는 것은 처음입니다. 판독 기준은 2026-11-18에 모델 응답이 생기기 전에 고정했고, 이 노트는 그 기준을 바꾸지 않습니다. 결과는 문장에 대한 민감도이며 정확도나 협상 성과가 아니고, 튜닝의 후보 선택에도 쓰이지 않습니다. 결과 파일은 상태 폴더의 `utterance_probe/` 아래 새 폴더에 저장되어 상태 ZIP에 함께 들어갑니다.


In [ ]:
#@title 8. (선택) 공급사 발언 진단 — 체크해야 실행
RUN_UTTERANCE_PROBE = False  #@param {type:"boolean"}
if not RUN_UTTERANCE_PROBE:
    print('RUN_UTTERANCE_PROBE가 꺼져 있어 발언 진단을 실행하지 않았습니다. 실행하려면 체크한 뒤 다시 누르세요.')
elif not NEG_STATE_READY or not NEG_READY:
    raise RuntimeError('1~4번의 소스·모델 준비를 먼저 완료하세요.')
elif not (NEG_STATE / 'protocol.json').is_file():
    raise RuntimeError('5번(계획 기록)을 먼저 실행하세요. 계획이 없는 상태 폴더에는 결과를 쓰지 않습니다.')
else:
    neg_utterance_dir = NEG_STATE / 'utterance_probe' / ('attempt-' + neg_stamp() + '-' + uuid.uuid4().hex[:4])
    neg_utterance_dir.mkdir(parents=True)        # a new folder every time; nothing is overwritten
    neg_utterance_file = neg_utterance_dir / 'result.json'
    neg_utterance_run = neg_run('negotiation/run_utterance_probe.py', *NEG_ARGS,
                                '--output', neg_utterance_file, check=False)
    neg_utterance = json.loads(neg_utterance_file.read_text(encoding='utf-8'))
    neg_reading = neg_utterance.get('verdict') or {}
    print('완료:', neg_utterance.get('complete'), '| 판독:', neg_reading.get('reading'),
          '| 모델 근거로 셈:', neg_reading.get('counts_as_model_evidence'))
    print('결과 파일:', neg_utterance_file)
    if neg_utterance_run['exit_code'] or not neg_utterance.get('complete'):
        raise RuntimeError('발언 진단이 완료되지 않았습니다. 실패한 요청의 값은 채우지 않았으며 판독에 쓰지 '
                           '않습니다. 위 오류와 결과 파일을 확인하세요.')
    print('문장에 대한 민감도이며 정확도나 협상 성과가 아닙니다. 판독 기준은 2026-11-18에 고정한 그대로입니다.')


### 9. 리더보드와 선택 결과 (모델 호출 없음)

완료된 실행만으로 표를 다시 만들고, 모든 후보가 끝난 상태(완료·사유 있는 실패·건너뜀)이면 선택을 한 번 기록합니다. 선택 규칙은 5번에서 고정한 그대로입니다: 선택 구간에서 완료되고 권한 위반과 검토 필요가 0건인 후보 중 평균 실현 절감이 가장 높은 것, 동률이면 앞선 후보(기본 구성 M0이 항상 우선).

- `cpu_reference_not_model_evidence` 행은 모델을 부르지 않는 CPU 기준이며 선택 대상이 아닙니다.
- 차이 열은 사례 단위 대응 부트스트랩의 평균과 90% 구간입니다. 구간에 0이 들어 있으면 낫다고 말할 수 없습니다.
- 선택 구간의 숫자는 고르는 데 쓴 표본 안의 값입니다. 주장은 10번의 확인 구간으로만 합니다.

**추정에 실패한 후보가 있을 때:** 이 셀은 표만 보여 주고 선택을 기록하지 않습니다. 실패를 받아들이고 나머지 후보 중에서 선택을 확정하려면 `ACCEPT_FAILED_CANDIDATE`를 체크하고 다시 실행하세요. 한 번 기록한 선택은 바뀌지 않습니다. 다시 시도하는 방법은 맨 아래 "막혔을 때"에 있습니다.


In [ ]:
#@title 9. 리더보드 · 선택 결과 보기 (모델 호출 없음)
ACCEPT_FAILED_CANDIDATE = False  #@param {type:"boolean"}
neg_waiting = neg_failed_waiting()
if neg_waiting and not ACCEPT_FAILED_CANDIDATE:
    print('추정에 실패한 후보가 있어 선택을 기록하지 않았습니다:', ', '.join(neg_waiting))
    print('아래는 마지막으로 만든 리더보드입니다. 실패를 받아들이고 나머지 후보 중에서 선택하려면 '
          'ACCEPT_FAILED_CANDIDATE를 체크하고 다시 실행하세요. 다시 시도하려면 "막혔을 때"를 보세요.')
else:
    neg_checked(neg_tune('board'), '리더보드')
neg_selection = neg_show_board()


### 10. 확인 구간 — 한 번만

선택한 구성(S)이 기본 구성(M0)보다 나은지를, 고르는 데 쓰지 않은 개발 사례 300건(dev-1250..1549)에서 **한 번만** 확인합니다. 실행을 시작하는 순간 첫 사용이 기록되고, 끝난 뒤에는 다시 실행해도 확인으로 인정되지 않습니다. 그래서 `RUN_CONFIRM`을 직접 체크해야만 실행됩니다. 9번에서 선택 결과를 확인한 뒤에 체크하세요.

- 1차 판정: S가 M0이 아니고, (S − M0) 실현 절감 차이의 90% 구간 하한이 0보다 크고, 평균 순절감 차이가 음수가 아니고, S에 권한 위반이 없을 때만 `improvement_confirmed`입니다. 그 밖에는 `not_confirmed`(기본 설정 유지)이고, 필요한 실행이 끝나지 않았으면 `incomplete`입니다.
- 시간 예산에 걸려 멈추면 같은 선택·같은 소스에서 다시 실행해 첫 사용을 이어 갈 수 있습니다.
- 첫 사용을 기록하기 전에 모델에 한 번 물어(이 답은 저장하지 않음) 저장된 답을 만든 것과 같은 실행 환경인지 확인합니다. 다르면 확인 구간을 쓰지 않고 멈춥니다.
- "한 번만"은 **이 상태 폴더 기준**입니다. 이전 상태 ZIP을 복원하거나 새 상태 폴더에서 다시 확인하면 프로그램이 알아채지 못합니다. 같은 확인 구간을 쓴 적이 있으면 그 사실을 결과와 함께 보고하세요.
- 이 구간도 개발 분할입니다. 판정이 좋게 나와도 보류 평가 결과나 실제 협상에 대한 근거가 아닙니다.


In [ ]:
#@title 10. 확인 구간 실행 (한 번만 · 체크해야 실행)
RUN_CONFIRM = False  #@param {type:"boolean"}
CONFIRM_BUDGET_MINUTES = 60  #@param {type:"number"}
if not RUN_CONFIRM:
    print('RUN_CONFIRM이 꺼져 있어 확인 구간을 실행하지 않았습니다. 확인 구간은 한 번만 쓸 수 있으므로 '
          '9번에서 선택 결과를 확인한 뒤 체크하세요.')
elif not NEG_STATE_READY or not NEG_READY:
    raise RuntimeError('1~4번의 소스·모델 준비를 먼저 완료하세요.')
elif not (NEG_STATE / 'selection.json').is_file():
    raise RuntimeError('선택 결과(selection.json)가 없습니다. 7번을 끝까지 실행하고 9번에서 선택을 확인한 뒤 '
                       '실행하세요. 확인 구간은 아직 사용하지 않았습니다.')
else:
    neg_confirm_budget = float(CONFIRM_BUDGET_MINUTES)
    if not 0 < neg_confirm_budget <= 720:
        raise ValueError('CONFIRM_BUDGET_MINUTES는 0보다 크고 720 이하인 분 단위 숫자여야 합니다.')
    neg_confirm = neg_stage('confirm', '--budget-minutes', neg_confirm_budget)
    if neg_confirm['status'] == 'partial_budget':
        print('확인 구간이 아직 끝나지 않았습니다. 이 셀을 다시 실행하면 첫 사용을 이어서 진행합니다.')
    neg_selection = neg_show_board()


### 11. 상태 ZIP 내려받기

상태 폴더 전체(프로토콜, 저장된 모델 답, 실행 기록, 리더보드, 선택)를 목록과 해시가 들어 있는 ZIP으로 만들어 내려받습니다. 합성 사례의 공개 요청, 모델 응답, 평가용 기록이 들어 있고 가중치와 로그인 정보는 들어 있지 않습니다. **런타임을 종료하기 전에 내려받으세요.** 오류를 담당자에게 전달할 때는 `DOWNLOAD_LOGS`를 체크해 명령 로그도 함께 받으세요.


In [ ]:
#@title 11. 상태 ZIP 만들기 · 내려받기
DOWNLOAD_LOGS = False  #@param {type:"boolean"}
if not NEG_STATE_READY:
    raise RuntimeError('1번 셀을 오류 없이 끝낸 뒤 실행하세요.')
neg_state_zip = neg_pack()
if neg_state_zip is None:
    raise RuntimeError('내려받을 상태가 없습니다. 5번(계획 기록)부터 실행하세요.')
from google.colab import files
files.download(str(neg_state_zip))
if DOWNLOAD_LOGS:
    import shutil
    neg_logs_zip = shutil.make_archive(
        str(NEG_PACKS / ('negotiation-tuning-logs-' + neg_stamp() + '-' + uuid.uuid4().hex[:4])),
        'zip', root_dir=NEG_LOGS)
    print('로그 ZIP:', neg_logs_zip)
    files.download(neg_logs_zip)
print('새 런타임에서 이어서 하려면 이 상태 ZIP 한 개를 파일 패널에 올리고 1번부터 실행하세요.')


## 12. 막혔을 때 · 로컬 Windows GPU PC에서 실행하기

### 막혔을 때

| 화면에 보이는 상황 | 할 일 |
|---|---|
| `EXPECTED_CONTENT_SHA256이 비어 있습니다` / 내용이 같은 소스 ZIP이 없음 | ZIP과 64자리 값을 같은 `make_source_bundle.py` 실행에서 받아 다시 넣습니다. 값을 모르면 ZIP을 풀지 않습니다. |
| GPU 없음 / 메모리 부족 | T4 선택을 확인하고, 같은 런타임의 다른 모델·서버를 종료한 뒤 2번부터 다시 실행합니다. |
| 모델 접근 거절 | 본인 계정의 Zoro-1 접근 승인과 3번 로그인을 확인합니다. |
| 설치 후 재시작 안내 | 런타임을 재시작하고 1번부터 진행합니다. 상태 폴더는 남아 있습니다. |
| 런타임이 끊기거나 사용 시간이 끝남 | 새 런타임에서 내려받아 둔 상태 ZIP 한 개를 파일 패널에 올리고 1번부터 실행합니다. 저장된 답은 다시 호출하지 않습니다. |
| `status=partial_budget` | 정상입니다. 같은 셀(7번 또는 10번)을 다시 실행합니다. |
| `status=interrupted` | 중단된 요청의 값은 저장되지 않았습니다. 같은 셀을 다시 실행하면 이어서 진행합니다. |
| `status=initialization_failed` | 모델을 올리지 못했습니다. 2~4번을 다시 확인하고 같은 셀을 다시 실행합니다. |
| `status=failed` (추정 실패) | 실패한 값은 채워지지 않았고 자동 재시도도 없습니다. 실행은 첫 실패에서 멈추며 오류 원문이 로그와 해당 실행의 `result.json`(`first_errors`)에 남습니다. 원인(로그, GPU 메모리)을 확인한 뒤 사람이 결정합니다. **그대로 이어 가기:** 7번 셀을 다시 실행하면 실패한 구성은 실패로 둔 채 나머지 구성을 실행합니다. **보정 구간이 재시도에도 실패할 때:** 새 코드 셀에서 `neg_tune('fit-map', '--declare-unavailable', '--reason', '<사유>')`로 맵을 만들 수 없음을 기록하면 M0·M1·M2만으로 계속합니다. **다시 시도:** 새 코드 셀에서 `neg_stage('all', '--budget-minutes', 45, '--retry-failed')`. **실패를 받아들임:** 9번에서 `ACCEPT_FAILED_CANDIDATE`를 체크합니다. |
| `status=store_write_failed` | 모델 답을 저장하지 못했습니다(디스크 공간 부족 등). 그 뒤로는 모델을 호출하지 않았습니다. 공간을 확보하고 같은 셀을 다시 실행합니다. |
| `another run is using this state directory` 거절 | 같은 상태 폴더로 실행 중인 다른 프로세스가 있습니다. 끝나기를 기다립니다. 그 프로세스가 이미 없다면 상태 폴더의 `run.lock`을 지우고 다시 실행합니다. |
| `status=smoke_failed` | 저장된 답만으로 같은 결과가 나오지 않았습니다. 다음 단계로 가지 말고 11번에서 상태 ZIP과 로그를 받아 담당자에게 전달합니다. |
| `status=store_identity_mismatch` 또는 `bound to store identity …` 거절 | 저장된 답을 만든 환경(소스 파일, 실행 옵션, GPU·dtype·라이브러리)과 지금이 다릅니다. 서로 다른 환경의 답은 섞지 않습니다. 새 상태 폴더로 시작합니다(아래). |
| `protocol digest differs` 거절 | 상태 폴더의 계획이 지금 소스의 프로토콜과 다릅니다. 새 상태 폴더로 시작합니다(아래). |
| `CONFIRM was already used` 거절 | 확인 구간은 한 번만 씁니다. 판정은 리더보드에 이미 있습니다. 다시 돌린 결과는 `not_confirmatory`로만 기록되며 판정을 바꾸지 않습니다. |

**새 상태 폴더로 시작하기:** 파일 패널에서 `/content/negotiation-tuning-state` 폴더의 이름을 바꿉니다(예: `negotiation-tuning-state-old`). `/content`에 올려 둔 상태 ZIP이 있으면 그 이름도 `negotiation-tuning-state`로 시작하지 않게 바꿉니다(예: `old-state.zip`). 그다음 1번부터 실행합니다. 이전 폴더와 내려받은 ZIP은 지우지 말고 보관하세요.

이 노트는 자체 Transformers 어댑터 경로입니다. 공식 SGLang 서버 경로가 아니며 두 경로의 결과를 같다고 보지 않습니다. 요청 해시와 응답 시간, 그리고 모델 응답에 들어 있는 GPU 이름·dtype·라이브러리 버전은 상태 폴더의 저장 기록과 각 결과 파일에 남습니다.

### 선택된 설정 파일

선택이 기록되면 상태 폴더에 `selected_config.json`이 생깁니다. 이후 로컬 PC에서 그 설정을 고정할 때 `negotiation/run_evaluation.py freeze --config-file`에 넘기는 파일입니다. 그 단계는 이 노트의 범위가 아니며 여기서는 실행하지 않습니다. CPU 대역이나 축소 블록에서 나온 선택은 모델 근거가 아닙니다.

### 로컬 Windows GPU PC (PowerShell)

아래 명령도 2026-11-18 기준으로 GPU에서 실행하지 않았습니다. 저장소 루트에서 실행합니다. `$zoro`는 `negotiation/docs/PLAN.md` 5절의 값 그대로이며 dtype은 `bf16`입니다. T4와 같은 `fp16`으로 실행하려면 `'--dtype','fp16'`으로 바꾸되, 한 상태 폴더 안에서는 옵션을 바꾸지 마세요. dtype이나 GPU가 다르면 저장된 답의 식별자나 기록된 실행 환경이 달라 실행기가 거절하므로 **Colab 상태 ZIP을 로컬에서 이어서 실행하지 말고 로컬 전용 상태 폴더를 쓰세요.**

```powershell
$gpuPython = '.cache/venv-gpu/Scripts/python.exe'
$zoro = @('--backend','zoro',
  '--original-checkpoint','.cache/hf/models--trillionlabs--Zoro-1/snapshots/766c5522cfb7df47dc9546df20fe3ce6efe6067c',
  '--exported-checkpoint','.cache/zoro-text-measured','--official-source-dir','.cache/trida-stack',
  '--device','cuda:0','--dtype','bf16','--decision-temperature','1','--max-input-tokens','2048')
$state = 'runs/negotiation/tuning/local-gpu'

# 5. 계획 기록 (모델 호출 없음)
& $gpuPython -X utf8 negotiation/run_tune.py plan --state-dir $state
# 6. 짧은 확인
& $gpuPython -X utf8 negotiation/run_tune.py run --stage smoke --state-dir $state @zoro
# 7. 전체 실행. 예산에 걸리면 같은 명령을 다시 실행
& $gpuPython -X utf8 negotiation/run_tune.py run --stage all --budget-minutes 45 --state-dir $state @zoro
# 8. (선택) 공급사 발언 진단. 실제 모델 결과는 덮어쓰지 않으므로 새 경로를 지정
& $gpuPython -X utf8 negotiation/run_utterance_probe.py @zoro --output runs/negotiation/utterance_zoro_local.json
# 9. 리더보드와 선택 (모델 호출 없음). 결과: $state/leaderboard.md, selection.json
& $gpuPython -X utf8 negotiation/run_tune.py board --state-dir $state
# 10. 확인 구간. 한 번만
& $gpuPython -X utf8 negotiation/run_tune.py run --stage confirm --budget-minutes 60 --state-dir $state @zoro
# 11. 상태 ZIP. 이미 있는 파일에는 쓰지 않으므로 새 이름을 지정
& $gpuPython -X utf8 negotiation/run_tune.py pack --state-dir $state --output runs/negotiation/tuning/local-gpu-state.zip
```

`run --stage all`이 `status=failed`로 끝났다면 `board`를 실행하기 전에 먼저 결정하세요. 명령줄의 `board`는 추정에 실패한 후보를 받아들이고 나머지 후보 중에서 선택을 확정하며, 확정한 선택은 바뀌지 않습니다. 다시 시도하려면 같은 `run --stage all` 명령에 `--retry-failed`를 붙입니다.

Colab에서 받은 상태 ZIP의 결과를 로컬에서 **보기만** 하려면 모델 없이 복원한 뒤 리더보드를 다시 만듭니다.

```powershell
& $gpuPython -X utf8 negotiation/run_tune.py restore --state-dir runs/negotiation/tuning/colab-t4 --input negotiation-tuning-state.zip
& $gpuPython -X utf8 negotiation/run_tune.py board --state-dir runs/negotiation/tuning/colab-t4
```

소스 ZIP을 만드는 명령(개발 PC, GPU 불필요)은 `python scripts/make_source_bundle.py`입니다. 커밋하지 않은 변경은 ZIP에 들어가지 않으며, 마지막 줄에 출력되는 값이 1번 셀에 넣는 내용 해시입니다.
